# Session 11 · Classification I

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Example 1 - the sigmoid

In [1]:
import numpy as np
import pandas as pd
z = np.array([-4, -1, 0, 1, 4])
print((1 / (1 + np.exp(-z))).round(3))

[0.018 0.269 0.5   0.731 0.982]


## Example 2 - logistic regression: pass / fail

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(11)
n = 400
hours = rng.uniform(0, 12, n)
attendance = rng.uniform(40, 100, n)
logit = -9 + 0.7 * hours + 0.08 * attendance
y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
X = np.column_stack([hours, attendance])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
logreg = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
print("test accuracy:", round(logreg.score(X_te, y_te), 3))
print("P(fail), P(pass) for 6 h & 85%:", logreg.predict_proba([[6, 85]]).round(3))

test accuracy: 0.75
P(fail), P(pass) for 6 h & 85%: [[0.133 0.867]]


## Example 3 - reading coefficients

In [3]:
coef = logreg[-1].coef_[0]
print(dict(zip(["study_hours", "attendance"], coef.round(2))))
assert (coef > 0).all()

{'study_hours': np.float64(2.67), 'attendance': np.float64(1.42)}


## Example 4 - changing the threshold

In [4]:
proba = logreg.predict_proba(X_te)[:, 1]
for t in (0.3, 0.5, 0.7):
    print(f"threshold {t}: predicted pass = {(proba >= t).sum()} of {len(proba)}")

threshold 0.3: predicted pass = 59 of 100
threshold 0.5: predicted pass = 46 of 100
threshold 0.7: predicted pass = 42 of 100


## Example 5 - KNN and the effect of k

In [5]:
from sklearn.neighbors import KNeighborsClassifier
for k in [1, 5, 15, 51]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_tr, y_tr)
    print(f"k={k:>2}: train {knn.score(X_tr, y_tr):.3f} | test {knn.score(X_te, y_te):.3f}")

k= 1: train 1.000 | test 0.730
k= 5: train 0.887 | test 0.790
k=15: train 0.880 | test 0.760
k=51: train 0.873 | test 0.750


## Example 6 - mixed numeric + categorical features

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
df = pd.DataFrame({"income": rng.normal(80, 25, n), "age": rng.integers(20, 60, n),
                   "city": rng.choice(["Karachi", "Lahore", "Quetta"], n)})
target = ((df["income"] > 75) ^ (rng.random(n) < 0.1)).astype(int)
pre = ColumnTransformer([("num", StandardScaler(), ["income", "age"]),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), ["city"])])
model = make_pipeline(pre, LogisticRegression(max_iter=1000))
d_tr, d_te, t_tr, t_te = train_test_split(df, target, random_state=0)
print("mixed-feature accuracy:", round(model.fit(d_tr, t_tr).score(d_te, t_te), 3))

mixed-feature accuracy: 0.93
